# **Step 1 : Dependencies Installation**

In [1]:
%pip install ultralytics torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


# **Step 2 : Model_Evalution** 

In [2]:
from ultralytics import YOLO

# Load model (explicit task)
model = YOLO(
    "/kaggle/input/20-epoch/pytorch/default/1/best (2).pt",
    task="detect"
)

# Run validation
metrics = model.val(
    data="/kaggle/input/kaggle-yaml/kaggle_new_data.yaml",
    imgsz=640,
    batch=16,
    device=0,          # single GPU only
    split="val",
    save_json=True,
    plots=True,
    verbose=True,
    visualize=True     # optional (slow)
)

# Print metrics
print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)
print("mAP75:", metrics.box.map75)
print("Per-class mAP:", metrics.box.maps)

# Confusion matrix as DataFrame
df_cm = metrics.confusion_matrix.to_df()
print(df_cm)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.252 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,848,445 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 1.0±0.6 ms, read: 43.9±19.0 MB/s, size: 549.3 KB)
val: Scanning /kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset/val/labels... 4196 images, 9 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4196/4196 153.2it/s 27.4s
WARNING ⚠️ val: Cache directory /kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset/val is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━

# **Step 3 : CSV File Creation**

In [3]:
import os

# Convert metrics to CSV string
val_csv = metrics.to_csv()
print(val_csv)

# Define output path
dir_path = "/kaggle/working/runs/detect"
csv_filename = "validation_results.csv"

# Ensure directory exists
os.makedirs(dir_path, exist_ok=True)

# Full file path
full_path = os.path.join(dir_path, csv_filename)

# Write CSV
with open(full_path, "w") as f:
    f.write(val_csv)

print("Saved at:", full_path)


Class,Images,Instances,Box-P,Box-R,Box-F1,mAP50,mAP50-95
animal,219,753,0.67127,0.25498,0.36958,0.30788,0.14234
autorickshaw,1436,3205,0.81907,0.64992,0.72476,0.72192,0.52165
bicycle,264,301,0.72806,0.45515,0.56013,0.49825,0.29517
bus,1061,1794,0.84061,0.62709,0.71832,0.69911,0.53721
car,2582,8793,0.83377,0.59553,0.6948,0.66711,0.47385
caravan,18,18,0.38456,0.72222,0.50188,0.48363,0.45113
motorcycle,2778,10059,0.79946,0.61557,0.69556,0.67238,0.40801
person,2105,8863,0.76899,0.41727,0.54099,0.49718,0.27436
rider,2454,9444,0.78676,0.49968,0.61119,0.57363,0.33783
traffic light,157,370,0.67994,0.28134,0.39799,0.34135,0.16665
traffic sign,774,1404,0.64645,0.33832,0.44418,0.36655,0.19861
train,4,4,1.0,0.0,0.0,0.00144,0.00029
truck,1564,2758,0.77989,0.60587,0.68195,0.67895,0.49204
vehicle fallback,1116,2072,0.61909,0.12934,0.21398,0.16563,0.09259

Saved at: /kaggle/working/runs/detect/validation_results.csv


# **Step 4 : Model BenchMarking**

In [4]:
from ultralytics.utils.benchmarks import benchmark

benchmark(
    model="/kaggle/input/20-epoch/pytorch/default/1/best (2).pt",
    data="/kaggle/input/kaggle-yaml/kaggle_new_data.yaml",
    imgsz=640,
    half=False,
    device=0,
)


Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6636.8/8062.4 GB disk)

Benchmarks complete for /kaggle/input/20-epoch/pytorch/default/1/best (2).pt on /kaggle/input/kaggle-yaml/kaggle_new_data.yaml at imgsz=640 (230.21s)
Benchmarks legend:  - ✅ Success  - ❎ Export passed but validation failed  - ❌️ Export failed
+----------------------------------------------------------------------------------------------------------+
|      Format                  Status❔   Size (MB)   metrics/mAP50-95(B)   Inference time (ms/im)   FPS   |
+==========================================================================================================+
| 1    PyTorch                 ✅         49.6        0.3137                14.05                    71.17 |
| 2    TorchScript             ❌         0.0         -                     -                        -     |
| 3    ONNX                    ❌         0.0         -                     -                        -     |
| 4    OpenVINO                ❌         0.0

,Format,Status❔,Size (MB),metrics/mAP50-95(B),Inference time (ms/im),FPS
"""1""","""PyTorch""","""✅""","""49.6""","""0.3137""","""14.05""","""71.17"""
"""2""","""TorchScript""","""❌""","""0.0""","""-""","""-""","""-"""
"""3""","""ONNX""","""❌""","""0.0""","""-""","""-""","""-"""
"""4""","""OpenVINO""","""❌""","""0.0""","""-""","""-""","""-"""
"""5""","""TensorRT""","""❌""","""0.0""","""-""","""-""","""-"""
"""6""","""CoreML""","""❌""","""0.0""","""-""","""-""","""-"""
"""7""","""TensorFlow SavedModel""","""❌""","""0.0""","""-""","""-""","""-"""
"""8""","""TensorFlow GraphDef""","""❌""","""0.0""","""-""","""-""","""-"""
"""9""","""TensorFlow Lite""","""❌""","""0.0""","""-""","""-""","""-"""
"""10""","""TensorFlow Edge TPU""","""❌""","""0.0""","""-""","""-""","""-"""


# **Step 5 : Zip File Creation** 

In [5]:
import shutil
import os

folder_path = "/kaggle/working/runs/detect"
zip_path = "/kaggle/working/runs/detect/evaluation_results"

# Remove existing zip if it exists
if os.path.exists(zip_path + ".zip"):
    os.remove(zip_path + ".zip")

# Create zip archive
shutil.make_archive(zip_path, 'zip', folder_path)

print("ZIP file created at:", zip_path + ".zip")

ZIP file created at: /kaggle/working/runs/detect/evaluation_results.zip


In [6]:
import os

file_path = "/kaggle/working/runs/detect/evaluation_results.zip"
print("Exists:", os.path.isfile(file_path))
print("Size (MB):", os.path.getsize(file_path) / (1024*1024))


Exists: True
Size (MB): 698.6617641448975


In [7]:
from IPython.display import FileLink

FileLink('/kaggle/working/runs/evaluation_results.zip')


/kaggle/working/runs/evaluation_results.zip

In [8]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/runs/evaluation_results.zip"))


/kaggle/working/runs/evaluation_results.zip

In [9]:
import os

file_path = "/kaggle/working/evaluation_results.zip"

if os.path.exists(file_path):
    print("✅ File exists:", file_path)
else:
    print("❌ File NOT found"ls
    !ls -lh /kaggle/working


SyntaxError: '(' was never closed (1495987015.py, line 8)